In [ ]:
import numpy as np
import pandas as pd
from hmmlearn.hmm import GaussianHMM
from filterpy.kalman import KalmanFilter
import matplotlib.pyplot as plt
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

class RegimeSwitchModel:
    def __init__(self, symbol, interval='1d', train_pct=0.7):
        self.symbol = symbol
        self.interval = interval
        self.train_pct = train_pct

    def compute_features(self, data, features_config):
        df = data.copy()
        for feature, config in features_config.items():
            if feature == 'log_return':
                df['log_return'] = np.log(df['Close']).pct_change()
            elif feature == 'lnrange':
                df['lnrange'] = np.log(df['High'] / df['Low'])
            elif feature == 'rsi':
                period = config.get('period', 14)
                delta = df['Close'].diff()
                gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
                loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
                rs = gain / loss
                df['rsi'] = 100 - (100 / (1 + rs))
        return df.dropna()

    def load_data(self):
        try:
            df_1 = pd.read_csv(f"/Users/valter.rebelo/MissionControl/data/micro/candleData/{self.symbol}_candles.csv")
            df_1['date'] = pd.to_datetime(df_1['date'])
            df_1.set_index('date', inplace=True)

            df_2 = pd.read_csv(f"/Users/valter.rebelo/MissionControl/data/micro/assetData/{self.symbol}.csv")
            df_2['date'] = pd.to_datetime(df_2['date'])
            df_2.set_index('date', inplace=True)

            btc_df = pd.read_csv("/Users/valter.rebelo/MissionControl/data/micro/candleData/bitcoin_candles.csv")
            btc_df['date'] = pd.to_datetime(btc_df['date'])
            btc_df.set_index('date', inplace=True)

            data = pd.merge(df_1, df_2[['total_volume', 'market_cap']], on='date', how='inner')
            data.rename(columns={'total_volume': 'Volume', 'open': 'Open', 'high': 'High', 'low': 'Low', 'close': 'Close'}, inplace=True)
            data.index.name = 'Date'

            if self.symbol != "bitcoin":
                data['close_btc'] = (data['Close'] / btc_df['close']) * 100
                data.dropna(inplace=True)

            if self.symbol == "bitcoin":
                data = data[data.index >= '2017-01-01']

            # Compute log_close for Kalman Filter
            data['log_close'] = np.log(data['Close'])

            # Compute dynamic features
            features_config = {
                'log_return': {},
                'lnrange': {},
                'rsi': {'period': 14}
            }
            data = self.compute_features(data, features_config)

            if len(data) < 100:
                raise ValueError("Insufficient data points (<100) for meaningful analysis.")
            logging.info(f"Loaded {len(data)} rows for {self.symbol}")
            return data
        except Exception as e:
            logging.error(f"Data loading failed for {self.symbol}: {str(e)}")
            raise

    def split_data(self, data, embargo_percent=0.01):
        try:
            total_rows = len(data)
            train_end_idx = int(total_rows * self.train_pct)
            embargo_end_idx = int(train_end_idx + (total_rows * embargo_percent))
            
            train = data.iloc[:train_end_idx]
            embargo = data.iloc[train_end_idx:embargo_end_idx]
            test = data.iloc[embargo_end_idx:]
            
            self.train_data = train
            if len(train) < 50 or len(test) < 50:
                raise ValueError("Train or test set too small (<50 rows).")
            logging.info(f"Split data: train={len(train)}, embargo={len(embargo)}, test={len(test)}")
            return train, embargo, test
        except Exception as e:
            logging.error(f"Data splitting failed: {str(e)}")
            raise

    def normalize_features(self, train, test, features):
        train_mean = train[features].mean()
        train_std = train[features].std()
        train_normalized = (train[features] - train_mean) / train_std
        test_normalized = (test[features] - train_mean) / train_std
        return (pd.DataFrame(train_normalized, index=train.index, columns=features),
                pd.DataFrame(test_normalized, index=test.index, columns=features))

    def train_hmm(self, train, features):
        try:
            f_train_normalized, _ = self.normalize_features(train, train, features)
            hmm = GaussianHMM(n_components=3, covariance_type='diag', n_iter=500, random_state=42)
            hmm.fit(f_train_normalized)
            if not hmm.monitor_.converged:
                logging.warning("HMM training did not converge.")
            logging.info("HMM trained successfully")
            return hmm
        except Exception as e:
            logging.error(f"HMM training failed: {str(e)}")
            raise

    def train_kf(self, test):
        try:
            kf = KalmanFilter(dim_x=2, dim_z=1)
            kf.F = np.array([[1, 1], [0, 1]])
            kf.H = np.array([[1, 0]])
            kf.Q = np.eye(2) * 0.01
            kf.R = np.array([[10]])
            kf.x = np.array([test['log_close'].iloc[0], 0])
            kf.P = np.eye(2) * 1000
            mu, _, _, _ = kf.batch_filter(test['log_close'].values)
            kf_slope_raw = np.diff(mu[:, 0])
            kf_slope = np.concatenate([[0], kf_slope_raw])
            logging.info("Kalman Filter trained successfully")
            return mu[:, 0], kf_slope
        except Exception as e:
            logging.error(f"Kalman Filter training failed: {str(e)}")
            raise

    def resample_weekly(self, data):
        weekly = data.resample('W-MON').agg({
            'Open': 'first', 'High': 'max', 'Low': 'min', 'Close': 'last', 
            'Volume': 'sum', 'log_close': 'last', 'log_return': 'sum', 
            'lnrange': 'mean', 'rsi': 'last'
        })
        return weekly.dropna()

    def train_models_multi_res(self, train, test, features):
        try:
            f_train_daily, f_test_daily = self.normalize_features(train, test, features)
            hmm_daily = GaussianHMM(n_components=3, covariance_type='diag', n_iter=50000, random_state=42)
            hmm_daily.fit(f_train_daily)
            kf_est_daily, kf_slope_daily = self.train_kf(test)

            train_weekly = self.resample_weekly(train)
            test_weekly = self.resample_weekly(test)
            f_train_weekly, f_test_weekly = self.normalize_features(train_weekly, test_weekly, features)
            hmm_weekly = GaussianHMM(n_components=3, covariance_type='diag', n_iter=50000, random_state=42)
            hmm_weekly.fit(f_train_weekly)
            kf_est_weekly, kf_slope_weekly = self.train_kf(test_weekly)

            logging.info("Multi-resolution models trained successfully")
            return (hmm_daily, f_test_daily, kf_slope_daily), (hmm_weekly, f_test_weekly, kf_slope_weekly)
        except Exception as e:
            logging.error(f"Multi-resolution training failed: {str(e)}")
            raise

    def voting_machine(self, hmm_daily, f_test_daily, kf_slope_daily, 
                       hmm_weekly, f_test_weekly, kf_slope_weekly, test_daily):
        hidden_states_daily = hmm_daily.predict(f_test_daily)
        fav_state_daily = np.argmax(hmm_daily.means_[:, 0])
        hmm_daily_state = ['Long' if s == fav_state_daily else 'Flat' for s in hidden_states_daily]
        kf_daily_state = ['Long' if s > 0 else 'Flat' if s <= 0 else 'Flat' for s in kf_slope_daily]
        logging.info(f"Daily predictions: {len(hmm_daily_state)} states")

        hidden_states_weekly = hmm_weekly.predict(f_test_weekly)
        fav_state_weekly = np.argmax(hmm_weekly.means_[:, 0])
        hmm_weekly_state = ['Long' if s == fav_state_weekly else 'Flat' for s in hidden_states_weekly]
        kf_weekly_state = ['Long' if s > 0 else 'Flat' if s <= 0 else 'Flat' for s in kf_slope_weekly]
        weekly_df_raw = pd.DataFrame({'hmm_weekly': hmm_weekly_state, 'kf_weekly': kf_weekly_state}, 
                                     index=f_test_weekly.index)
        weekly_df = weekly_df_raw.reindex(test_daily.index[1:], method='ffill')
        logging.info(f"Weekly predictions: {len(hmm_weekly_state)} states, reindexed to {len(weekly_df)} rows")

        if len(hmm_daily_state) != len(weekly_df):
            logging.warning(f"Length mismatch: daily={len(hmm_daily_state)}, weekly={len(weekly_df)}. Using minimum length.")

        ens_state = []
        min_len = min(len(hmm_daily_state), len(weekly_df))
        for i in range(min_len):
            daily_vote = hmm_daily_state[i] == 'Long' and kf_daily_state[i] == 'Long'
            weekly_vote = weekly_df['hmm_weekly'].iloc[i] == 'Long' and weekly_df['kf_weekly'].iloc[i] == 'Long'
            if daily_vote and weekly_vote:
                ens_state.append('Long')
            elif not daily_vote and not weekly_vote:
                ens_state.append('Flat')
            else:
                ens_state.append('Long' if daily_vote else 'Flat')
        return pd.Series(ens_state, index=test_daily.index[1:min_len+1])

    def ensemble_predict(self, hmm, test, kf_slope, features):
        try:
            models_daily, models_weekly = self.train_models_multi_res(self.train_data, test, features)
            states = self.voting_machine(*models_daily, *models_weekly, test)
            logging.info("Ensemble prediction completed")
            return states
        except Exception as e:
            logging.error(f"Ensemble prediction failed: {str(e)}")
            raise

    def simulate_trading(self, test, states):
        try:
            shifted_states = states.shift(1).fillna('Flat')
            returns = test['Open'].pct_change() * (shifted_states == 'Long').astype(int)
            cum_returns = (1 + returns).cumprod()
            result = pd.DataFrame({'Close': test['Close'], 'ens_state': states, 
                                  'returns': returns, 'cum_returns': cum_returns})
            logging.info("Trading simulation completed")
            return result
        except Exception as e:
            logging.error(f"Trading simulation failed: {str(e)}")
            raise

    def evaluate(self, results):
        try:
            logging.info(f"ens_state sample: {results['ens_state'].head().tolist()}")
            cum_returns = results['cum_returns'].fillna(1.0)
            logging.info(f"Final cum_returns: {cum_returns.iloc[-1]}, length: {len(results)}")
            ann_ret = (cum_returns.iloc[-1] ** (252/len(results))) - 1
            returns = results['returns'].fillna(0.0)
            sharpe = (returns.mean() / returns.std()) * np.sqrt(252)
            ens_state = results['ens_state']
            switches = sum(ens_state.iloc[i] != ens_state.iloc[i-1] for i in range(1, len(ens_state)))
            metrics = {'Annualized Return': ann_ret, 'Sharpe Ratio': sharpe, 'Switches': switches}
            logging.info(f"Evaluation metrics: {metrics}")
            return metrics
        except Exception as e:
            logging.error(f"Evaluation failed: {str(e)}")
            raise



In [ ]:
# Usage
if __name__ == "__main__":
    model = RegimeSwitchModel('solana')
    try:
        data = model.load_data()
        train, embargo, test = model.split_data(data)
        np.random.seed(42)
        
        features = ['log_return', 'lnrange', 'rsi']
        states = model.ensemble_predict(None, test, None, features)
        print("State Distribution:")
        print(states.value_counts())
        
        results = model.simulate_trading(test, states)
        metrics = model.evaluate(results)
        print("Performance Metrics:", metrics)
        print("Last 5 rows of results:")
        print(results.tail())

        plt.plot(results.index, results['cum_returns'])
        plt.title(f"{model.symbol} Cumulative Returns")
        plt.show()
    except Exception as e:
        logging.error(f"Pipeline execution failed: {str(e)}")
        raise

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(go.Scatter(x=results.index, y=results['cum_returns'], mode='lines', name='Cumulative Returns'))
fig.update_layout(title=f"{model.symbol} Cumulative Returns",
                 xaxis_title="Date",
                 yaxis_title="Cumulative Returns")
fig.show()